In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchaudio import datasets, transforms,load
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torch.utils.data import DataLoader,Dataset,random_split
import torch.nn.functional as F
import pandas as pd
import os


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("chrisfilo/urbansound8k")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'urbansound8k' dataset.
Path to dataset files: /kaggle/input/urbansound8k


In [3]:
data_path = '/kaggle/input/urbansound8k'

In [4]:
csv_path = '/kaggle/input/urbansound8k/UrbanSound8K.csv'

In [5]:
df = pd.read_csv(csv_path)
df.head()

,slice_file_name,fsID,start,end,salience,fold,classID,class
0,100032-3-0-0.wav,100032,0.0,0.317551,1,5,3,dog_bark
1,100263-2-0-117.wav,100263,58.5,62.500000,1,5,2,children_playing
2,100263-2-0-121.wav,100263,60.5,64.500000,1,5,2,children_playing
3,100263-2-0-126.wav,100263,63.0,67.000000,1,5,2,children_playing
4,100263-2-0-137.wav,100263,68.5,72.500000,1,5,2,children_playing


In [6]:
label = sorted(df['class'].unique())
label

['air_conditioner',
 'car_horn',
 'children_playing',
 'dog_bark',
 'drilling',
 'engine_idling',
 'gun_shot',
 'jackhammer',
 'siren',
 'street_music']

In [7]:
len(label)

10

In [8]:
label_to_index = {word: ind for ind,word in enumerate(label)}

In [9]:
label_to_index

{'air_conditioner': 0,
 'car_horn': 1,
 'children_playing': 2,
 'dog_bark': 3,
 'drilling': 4,
 'engine_idling': 5,
 'gun_shot': 6,
 'jackhammer': 7,
 'siren': 8,
 'street_music': 9}

In [10]:
transform = transforms.MelSpectrogram(sample_rate=22050,n_mels=64)

In [11]:
max_len = 500

In [12]:
class UrbanDataset(Dataset):
    def __init__(self,data_path,transform,max_len):
      self.data_path = data_path
      self.transform = transform
      self.max_len = max_len
      self.audios = []

      for name in os.listdir(data_path):
        if name.endswith('.csv'):
          continue

        name_path = os.path.join(data_path,name)
        for file in os.listdir(name_path):
          if file.endswith('.wav'):
            file_path = os.path.join(name_path, file)
            try:
              classes = list(df[df['slice_file_name'] == file]['class'])[0]
              self.audios.append((file_path,classes))
            except Exception as e:
              print(f'Ошибка {e}')
    def __len__(self,):
      return len(self.audios)

    def __getitem__(self,ind):
      file_path, classes = self.audios[ind]
      waveform, sr = load(file_path)

      # Приведение к моно (1 канал)
      if waveform.shape[0] > 1:
          waveform = torch.mean(waveform, dim=0, keepdim=True)

      if sr != 22050:
          resample = transforms.Resample(orig_freq=sr,new_freq= 22050)
          waveform = resample(waveform)

      spec = self.transform(waveform).squeeze(0)

      if spec.shape[1] > self.max_len:
        spec = spec[:, :self.max_len]

      if spec.shape[1] < self.max_len:
        count_len = self.max_len - spec.shape[1]
        spec = F.pad(spec, (0, count_len))

      return spec, label_to_index[classes]

In [13]:
datasets = UrbanDataset(data_path,transform,max_len)

In [14]:
train_size = int(len(datasets) * 0.8)
test_size = len(datasets) - train_size

train_data, test_data = random_split(datasets, [train_size, test_size],generator=torch.Generator().manual_seed(42))

In [15]:
train = DataLoader(train_data, batch_size=32, shuffle=True)
test = DataLoader(test_data, batch_size=32)

In [16]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [17]:
class UrbanAudio(nn.Module):
  def __init__(self):
    super().__init__()
    self.first = nn.Sequential(
        nn.Conv2d(1,16,kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.Conv2d(16,32,kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),
        nn.AdaptiveAvgPool2d((8,8))

    )
    self.second = nn.Sequential(
        nn.Flatten(),
        nn.Linear(32 * 8 * 8, 128),
        nn.ReLU(),
        nn.Linear(128,10),
    )
  def forward(self, x):
    x = x.unsqueeze(1)
    x = self.first(x)
    x = self.second(x)
    return x

In [18]:
model = UrbanAudio().to(device)

In [19]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [20]:
for epoch in range(15):
  model.train()
  total_loss = 0

  for x_batch, y_batch in train:
    x_batch, y_batch = x_batch.to(device), y_batch.to(device)

    y_pred = model(x_batch)
    loss = loss_fn(y_pred, y_batch)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss += loss.item()
  print(f'Эпоха {epoch+1}, потери: {total_loss:.2f}')

Эпоха 1, потери: 416.99
Эпоха 2, потери: 305.00
Эпоха 3, потери: 248.12
Эпоха 4, потери: 216.00
Эпоха 5, потери: 183.37
Эпоха 6, потери: 173.42
Эпоха 7, потери: 161.07
Эпоха 8, потери: 157.70
Эпоха 9, потери: 136.40
Эпоха 10, потери: 120.51
Эпоха 11, потери: 114.76
Эпоха 12, потери: 100.67
Эпоха 13, потери: 123.95
Эпоха 14, потери: 104.51
Эпоха 15, потери: 107.06


In [21]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
  for x_batch, y_batch in test:
    x_batch, y_batch = x_batch.to(device), y_batch.to(device)
    y_pred = model(x_batch)
    pred = torch.argmax(y_pred, dim=1)

    correct += (pred == y_batch).sum().item()
    total += y_batch.size(0)

accuracy = 100 * correct / total
print(f'Точность модели: {accuracy}%')

Точность модели: 74.52776187750429%


In [24]:
torch.save(model.state_dict(), 'urban_model.pth')
torch.save(label_to_index, 'urban_labels.pth')